# 46 · 全链路追踪、故障与课程复盘

单元：S17,S18。先修：35–45以及25–30。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：构造链路API与正式SciFact API分开；复用同一服务。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 正常链路
输入→规范化/改写→意图→词法/机制语义候选→候选缓存→字段重排。追踪每步，机制语义通道不是27章神经编码器。

In [2]:
cache=CandidateCache()
t=pipeline('cellphone repair',cache); display(t)
assert t['query']=='phone repair' and t['ranking']

{'query': 'phone repair',
 'intent': 'support',
 'lexical': ['c', 'd', 'a', 'b'],
 'semantic': ['a', 'c', 'd'],
 'cache': 'miss',
 'candidates': ['a', 'b', 'c', 'd'],
 'ranking': ['c', 'd', 'a', 'b'],
 'fallback': None,
 'empty': False}

## 命中与降级
只改变故障标志；调用方必须能识别lexical fallback。缓存key需带渠道/模型版本，否则故障后可能复用不同策略缓存。此演示为不同策略使用独立cache。

In [3]:
print(pipeline('phone repair',cache,now=1))
fallback=pipeline('phone repair',fail_semantic=True)
print(fallback)
assert fallback['fallback']=='lexical'
empty=pipeline('unfindable'); assert empty['empty']; print(empty)

{'query': 'phone repair', 'intent': 'support', 'lexical': ['c', 'd', 'a', 'b'], 'semantic': ['a', 'c', 'd'], 'cache': 'hit', 'candidates': ['a', 'b', 'c', 'd'], 'ranking': ['c', 'd', 'a', 'b'], 'fallback': None, 'empty': False}
{'query': 'phone repair', 'intent': 'support', 'lexical': ['c', 'd', 'a', 'b'], 'semantic': [], 'cache': 'miss', 'candidates': ['a', 'b', 'c', 'd'], 'ranking': ['c', 'd', 'a', 'b'], 'fallback': 'lexical', 'empty': False}
{'query': 'unfindable', 'intent': 'abstain', 'lexical': [], 'semantic': [], 'cache': 'miss', 'candidates': [], 'ranking': [], 'fallback': None, 'empty': True}


## 同一FastAPI教学入口
/api/search/workshop/trace显式标构造，不替换/api/search正式模型。服务端每次独立cache以防跨请求状态误解，Notebook前两格才测跨请求缓存。

In [4]:
import urllib.request,json
payload=json.dumps({'query':'battrey repair','fail_semantic':True}).encode()
req=urllib.request.Request('http://127.0.0.1:18580/api/search/workshop/trace',data=payload,headers={'Content-Type':'application/json'})
with urllib.request.urlopen(req,timeout=20) as response: result=json.load(response)
assert result['evidence']=='constructed mechanism' and result['trace']['fallback']=='lexical'
print(result)

{'evidence': 'constructed mechanism', 'scope': 'Four artificial documents; rule semantic substitute; request-local cache; official SciFact service unchanged.', 'trace': {'query': 'battery repair', 'intent': 'support', 'lexical': ['c', 'd', 'a', 'b'], 'semantic': [], 'cache': 'miss', 'candidates': ['a', 'b', 'c', 'd'], 'ranking': ['c', 'd', 'a', 'b'], 'fallback': 'lexical', 'empty': False}}


## 复盘矩阵
按需求/判断→查询→召回→推理训练→排序→推荐→服务复盘。17是综合参考，不是前置。旧300测试仅诊断，不能用来调新模型。

In [5]:
from pathlib import Path
curriculum=json.loads(Path('docs/curriculum.json').read_text())
for row in curriculum['search_units']: print(row['id'],row['topic'],row['chapters'])
assert len(curriculum['search_units'])==18

S01 任务、用户需求与搜索产品 [35]
S02 相关性分档、标注与评估设计 [35, 25]
S03 文本规范化、中文分词与新词 [36]
S04 查询词权重与字段匹配 [37]
S05 类目与意图识别 [38]
S06 查询改写、纠错与语义漂移 [39]
S07 倒排索引、布尔与位置约束 [40]
S08 词法相关性与BM25 [26]
S09 语义表示与向量召回 [27]
S10 近似索引与候选预算 [28]
S11 缓存召回与一致性 [41]
S12 BERT相关性打分与推理 [42, 29]
S13 相关性模型训练 [42]
S14 排序目标与多路融合训练 [43, 29]
S15 质量、多样性与业务约束 [44]
S16 查询推荐与建议词 [45]
S17 服务链路、误差定位与性能 [46, 30]
S18 综合复盘与课程导航 [46]


## 本次保存结果的解读

cellphone被纠正为phone后可追溯到各路候选、缓存和排名；第二次调用命中缓存，语义通道故障明确标lexical fallback，未知词得到empty=true。同一服务的教学端点响应constructed mechanism；这些四文档机制不能替代30章的真实SciFact服务一致性证据。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。